In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\oscar\OneDrive\Escritorio\SOURCES TGF\mi_proyecto


In [10]:
import pandas as pd
from pathlib import Path

path_bank = Path(r"D:\tfg_finance\mi_proyecto_datos\local_db\raw\bank_list_MES.csv")

raw = pd.read_csv(path_bank, header=None, sep="\n", engine="python")
print(raw.head(15))

ValueError: Specified \n as separator or delimiter. This forces the python engine which does not accept a line terminator. Hence it is not allowed to use the line terminator as separator.

In [6]:
banks = pd.read_csv(
    path_bank,
    sep=",",           # separador explícito
    engine="python",   # parser más tolerante
    on_bad_lines="warn"  # o "skip" si quieres saltar filas problemáticas
)
banks.columns = banks.columns.str.strip()
banks.head()

C:\Users\oscar\AppData\Local\Temp\ipykernel_30496\3355045404.py:1: ParserWarning: Skipping line 11: Expected 3 fields in line 11, saw 4

  banks = pd.read_csv(
C:\Users\oscar\AppData\Local\Temp\ipykernel_30496\3355045404.py:1: ParserWarning: Skipping line 17: Expected 3 fields in line 17, saw 4

  banks = pd.read_csv(
C:\Users\oscar\AppData\Local\Temp\ipykernel_30496\3355045404.py:1: ParserWarning: Skipping line 24: Expected 3 fields in line 24, saw 4

  banks = pd.read_csv(
C:\Users\oscar\AppData\Local\Temp\ipykernel_30496\3355045404.py:1: ParserWarning: Skipping line 25: Expected 3 fields in line 25, saw 4

  banks = pd.read_csv(
C:\Users\oscar\AppData\Local\Temp\ipykernel_30496\3355045404.py:1: ParserWarning: Skipping line 26: Expected 3 fields in line 26, saw 4

  banks = pd.read_csv(
C:\Users\oscar\AppData\Local\Temp\ipykernel_30496\3355045404.py:1: ParserWarning: Skipping line 34: Expected 3 fields in line 34, saw 4

  banks = pd.read_csv(
C:\Users\oscar\AppData\Local\Temp\ipyker

,,rssd_id;name;Asset_share_100;Market_Cap_bll;ticket
1039502;JPMORGAN CHASE & CO.;0,1294;183,150;JPM
1120754;WELLS FARGO & COMPANY;0,0742;155,800;WFC
1951350;CITIGROUP INC.;0,1213;131,620;C
1073757;BANK OF AMERICA CORPORATION;0,1418;127,530;BAC
1119794;U.S. BANCORP;0,0171;49,700;USB


In [1]:
import pandas as pd
from pathlib import Path

path_bank = Path(r"D:\tfg_finance\mi_proyecto_datos\local_db\raw\bank_list_MES.csv")

banks = pd.read_csv(
    path_bank,
    sep=";",           # separador de columnas
    engine="python"
)

banks.columns = banks.columns.str.strip()
print(banks.head())
print(banks.columns.tolist())

   rssd_id ticket  Asset_share_100  Market_Cap_bll
0  1039502    JPM            12.94           183.0
1  1120754    WFC             7.42           156.0
2  1951350      C            12.13           132.0
3  1073757    BAC            14.18           128.0
4  1119794    USB             1.71            50.0
['rssd_id', 'ticket', 'Asset_share_100', 'Market_Cap_bll']


In [2]:
import yfinance as yf
import pandas as pd
from pathlib import Path

# Ya tienes esto hecho:
# path_bank = Path(r"D:\tfg_finance\mi_proyecto_datos\local_db\raw\bank_list_MES.csv")
# banks = pd.read_csv(path_bank, sep=";", engine="python")

tickers = banks["ticket"].dropna().unique().tolist()

start = "1996-01-01"
end   = "2026-01-01"

data = yf.download(
    tickers=tickers,
    start=start,
    end=end,
    interval="1mo",      # frecuencia mensual
    auto_adjust=False,   # usa True si quieres precios ajustados
    progress=False,
    group_by="ticker"
)

$NATIONAL CITY CORPORATION: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: COMMERCE BANCORP, INC."}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: UNIONBANCAL CORPORATION"}}}
$WACHOVIA CORPORATION: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1mo 1996-01-01 -> 2026-01-01) (Yahoo error = "No data found, symbol may be delisted")
$UNIONBANCAL CORPORATION: possibly delisted; no timezone found
$COMMERCE BANCORP, INC.: possibly delisted; no timezone found

5 Failed downloads:
['NATIONAL CITY CORPORATION', 'WACHOVIA CORPORATION', 'UNIONBANCAL CORPORATION', 'COMMERCE BANCORP, INC.']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1mo 1996-01-01 -> 2026-01-01) (Yahoo error = "No data found, symbol may be delisted")


In [15]:
# Lista completa de tickets que tienes
banks["ticket"].unique()

array(['JPM', 'WFC', 'C', 'BAC', 'USB', 'BK', 'PNC', 'STT', 'BGS', nan,
       'FITB', 'NTRS', 'MTB', 'RF', 'KEY', 'HBAN', 'ZION', 'BOKF', 'CBSH',
       'CFR', 'BPOP', 'FHN', 'ASB', 'TFSL', 'FULT', 'BOH', 'VLY', 'UMBF',
       'TRMK', 'CATY', 'FFIN', 'UBSI', 'GBCI', 'PRK', 'CVBF', 'FFBC',
       'NBTB', 'COLB', 'IBCP', 'BANF', 'CPF', 'WSBC', 'CHCO', 'SRCE',
       'NATIONAL CITY CORPORATION', 'WACHOVIA CORPORATION',
       'COMMERCE BANCORP, INC.', 'UNIONBANCAL CORPORATION'], dtype=object)

In [16]:
bad_tickers = [
    "UNIONBANCAL CORPORATION",
    "COMMERCE BANCORP, INC.",
    "NATIONAL CITY CORPORATION",
    "WACHOVIA CORPORATION",
]

In [17]:
banks_clean = banks[~banks["ticket"].isin(bad_tickers)].copy()

tickers = banks_clean["ticket"].dropna().unique().tolist()
print(tickers[:20])

['JPM', 'WFC', 'C', 'BAC', 'USB', 'BK', 'PNC', 'STT', 'BGS', 'FITB', 'NTRS', 'MTB', 'RF', 'KEY', 'HBAN', 'ZION', 'BOKF', 'CBSH', 'CFR', 'BPOP']


In [18]:
import yfinance as yf

start = "1996-01-01"
end   = "2026-01-01"

data = yf.download(
    tickers=tickers,
    start=start,
    end=end,
    interval="1mo",
    auto_adjust=False,
    progress=True,
    group_by="ticker"
)

[*********************100%***********************]  43 of 43 completed


In [20]:
print(type(data))
print(data.head())
print(data.columns)

<class 'pandas.core.frame.DataFrame'>
Ticker          WFC                                                 BGS       \
Price          Open     High      Low    Close Adj Close    Volume Open High   
Date                                                                           
1996-01-01  8.28125  8.59375  7.62500  8.59375  3.838019  62991200  NaN  NaN   
1996-02-01  8.59375  9.25000  8.21875  9.12500  4.104248  53405200  NaN  NaN   
1996-03-01  9.15625  9.28125  8.28125  9.18750  4.132358  63993200  NaN  NaN   
1996-04-01  9.21875  9.37500  8.59375  9.03125  4.062079  54786800  NaN  NaN   
1996-05-01  8.96875  9.25000  8.37500  8.71875  3.921525  54781200  NaN  NaN   

Ticker                ...      CBSH                                     JPM  \
Price      Low Close  ...       Low     Close Adj Close   Volume       Open   
Date                  ...                                                     
1996-01-01 NaN   NaN  ...  7.338274  7.415791  4.056334  3472949  19.500000   
1996-

In [4]:
import pandas as pd

# data: DataFrame que te devolvió yf.download, con MultiIndex en columnas
# niveles: 0 = Ticker, 1 = Price

# Seleccionamos todas las columnas cuyo nivel 1 es 'Adj Close'
adj_close = (
    data.xs("Adj Close", axis=1, level="Price")  # columnas = tickers
    .reset_index()
    .rename(columns={"Date": "date"})
)

# Ancho → largo: date, ticket, price_adj
prices_long = adj_close.melt(
    id_vars="date",
    var_name="ticket",
    value_name="price_adj"
).dropna(subset=["price_adj"])

prices_long.head()
prices_long.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14849 entries, 135 to 15479
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       14849 non-null  datetime64[ns]
 1   ticket     14849 non-null  object        
 2   price_adj  14849 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 464.0+ KB


In [22]:
panel_prices = (
    prices_long.merge(banks_clean, on="ticket", how="left")
    .sort_values(["rssd_id", "date"])
    .reset_index(drop=True)
)

panel_prices.head()
panel_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15192 entries, 0 to 15191
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             15192 non-null  datetime64[ns]
 1   ticket           15192 non-null  object        
 2   price_adj        15192 non-null  float64       
 3   rssd_id          15192 non-null  int64         
 4   Asset_share_100  15192 non-null  float64       
 5   Market_Cap_bll   15192 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 712.3+ KB


In [23]:
from pathlib import Path

out_path = Path(
    r"D:\tfg_finance\mi_proyecto_datos\local_db\processed\chicago_fed\bank_list_MES-prices-monthly-1996_2026.parquet"
)

panel_prices.to_parquet(out_path, index=False)
print(out_path)

D:\tfg_finance\mi_proyecto_datos\local_db\processed\chicago_fed\bank_list_MES-prices-monthly-1996_2026.parquet


In [25]:
import pandas as pd

sp_fin = pd.read_csv(r"D:\tfg_finance\mi_proyecto_datos\local_db\raw\SP_financial_index_1996-2025.csv")

sp_fin.columns = sp_fin.columns.str.strip()

# Aseguramos que la columna de año se llama 'year'
sp_fin = sp_fin.rename(columns={"Date": "year"})

# Hacemos melt de las 12 columnas de meses
month_cols = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

sp_long = sp_fin.melt(
    id_vars="year",
    value_vars=month_cols,
    var_name="month",
    value_name="sp_financial"
)

sp_long.head()

KeyError: "The following id_vars or value_vars are not present in the DataFrame: ['year', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']"

In [28]:
import pandas as pd
from pathlib import Path

# 1. Leer el fichero (ajusta la ruta y el separador si hace falta)
path_sp = Path(r"D:\tfg_finance\mi_proyecto_datos\local_db\raw\SP_financial_index_1996-2025.csv")

sp_fin = pd.read_csv(path_sp, sep=";", engine="python")
sp_fin.columns = sp_fin.columns.str.strip()

# Renombrar 'Date' -> 'year'
sp_fin = sp_fin.rename(columns={"Date": "year"})

month_cols = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

sp_long = sp_fin.melt(
    id_vars="year",
    value_vars=month_cols,
    var_name="month",
    value_name="sp_financial"
)

In [29]:
# Reemplazar coma decimal por punto y convertir a float
sp_long["sp_financial"] = (
    sp_long["sp_financial"]
    .astype(str)
    .str.replace(".", "", regex=False)   # por si hubiera separador de miles tipo 1.234,56
    .str.replace(",", ".", regex=False)  # coma decimal -> punto
    .astype(float)
)

In [30]:
month_map = {
    "January":   1,
    "February":  2,
    "March":     3,
    "April":     4,
    "May":       5,
    "June":      6,
    "July":      7,
    "August":    8,
    "September": 9,
    "October":   10,
    "November":  11,
    "December":  12,
}

sp_long["month_num"] = sp_long["month"].map(month_map)
sp_long["year"] = sp_long["year"].astype(int)

# Fecha al primer día del mes
sp_long["date"] = pd.to_datetime(
    dict(year=sp_long["year"], month=sp_long["month_num"], day=1)
)

# Llevarla al último día de mes
sp_long["date"] = sp_long["date"] + pd.offsets.MonthEnd(0)

# Orden y columnas finales
sp_long = sp_long.sort_values("date").reset_index(drop=True)
sp_long = sp_long[["date", "sp_financial"]]

sp_long.head()

,date,sp_financial
0,1996-01-31,163.00
1,1996-02-29,165.72
2,1996-03-31,167.35
3,1996-04-30,164.00
4,1996-05-31,167.20


In [31]:
out_sp = Path(
    r"D:\tfg_finance\mi_proyecto_datos\local_db\processed\indices\sp_financial-monthly-1996_2026.parquet"
)
out_sp.parent.mkdir(parents=True, exist_ok=True)

sp_long.to_parquet(out_sp, index=False)
print(out_sp)

D:\tfg_finance\mi_proyecto_datos\local_db\processed\indices\sp_financial-monthly-1996_2026.parquet


In [27]:
import pandas as pd
from pathlib import Path

path_sp = Path(r"D:\tfg_finance\mi_proyecto_datos\local_db\raw\SP_financial_index_1996-2025.csv")

# Si el fichero está en formato europeo, prueba con ; y decimal=,
sp_fin = pd.read_csv(path_sp, sep=";", engine="python")

print(sp_fin.columns.tolist())
print(sp_fin.head())

['Date', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
   Date January February   March   April     May    June    July  August  \
0  1996  163,00   165,72  167,35  164,00  167,20  168,70  164,86  170,08   
1  1997  212,46   220,49  204,49  218,76  228,63  240,95  269,24  248,85   
2  1998  312,32   341,24  360,25  365,62  356,44  370,88  370,43  284,68   
3  1999  357,03   361,35  374,57  399,45  376,84  391,80  366,92  349,67   
4  2000  318,93   283,92  335,90  324,78  345,89  324,59  357,63  391,17   

  September October November December  
0    181,34  194,47   212,79   204,72  
1    268,56  262,55   272,63   285,85  
2    289,94  324,42   346,25   352,76  
3    330,93  385,53   366,22   358,27  
4    400,21  397,93   373,70   407,18  


In [32]:
import yfinance as yf
import pandas as pd
from pathlib import Path

# tickers limpios
tickers = banks_clean["ticket"].dropna().unique().tolist()

start = "1996-01-01"
end   = "2026-01-01"

data_daily = yf.download(
    tickers=tickers,
    start=start,
    end=end,
    interval="1d",      # frecuencia diaria
    auto_adjust=False,  # True si quieres precios ajustados
    progress=True,
    group_by="ticker"
)

[*********************100%***********************]  43 of 43 completed


In [33]:
# data_daily: MultiIndex columns: nivel 0 = ticker, nivel 1 = Price (Open, High, ..., Adj Close, Volume)

adj_close_daily = (
    data_daily.xs("Adj Close", axis=1, level="Price")  # columnas = tickers
    .reset_index()
    .rename(columns={"Date": "date"})
)

# Ancho → largo: date, ticket, price_adj
prices_daily_long = adj_close_daily.melt(
    id_vars="date",
    var_name="ticket",
    value_name="price_adj"
).dropna(subset=["price_adj"])

prices_daily_long.head()
prices_daily_long.info()

<class 'pandas.core.frame.DataFrame'>
Index: 318938 entries, 2867 to 324649
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   date       318938 non-null  datetime64[ns]
 1   ticket     318938 non-null  object        
 2   price_adj  318938 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 9.7+ MB


In [34]:
panel_prices_daily = (
    prices_daily_long.merge(banks_clean, on="ticket", how="left")
    .sort_values(["rssd_id", "date"])
    .reset_index(drop=True)
)

panel_prices_daily.head()
panel_prices_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318938 entries, 0 to 318937
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   date             318938 non-null  datetime64[ns]
 1   ticket           318938 non-null  object        
 2   price_adj        318938 non-null  float64       
 3   rssd_id          318938 non-null  int64         
 4   Asset_share_100  318938 non-null  float64       
 5   Market_Cap_bll   318938 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 14.6+ MB


In [35]:
out_path_daily = Path(
    r"D:\tfg_finance\mi_proyecto_datos\local_db\processed\chicago_fed\bank_list_MES-prices-daily-1996_2026.parquet"
)

panel_prices_daily.to_parquet(out_path_daily, index=False)
print(out_path_daily)

D:\tfg_finance\mi_proyecto_datos\local_db\processed\chicago_fed\bank_list_MES-prices-daily-1996_2026.parquet
